# Humanoid retargeting: the paper table from per-sequence results

This notebook rebuilds **Table "Humanoid Retargeting"** (`tables/exp_retgt.tex`) of the WMM paper.

* Input: `release_data/retargeting/L1_results/` (13 MB). Set the environment variable `WMM_DATA` to use another data root.
* Each `reports/*.xlsx` file has **one row per test sequence** (2000 AMASS clips) for one (method, setting) pair. It was written by the scorer `eval.py` after the physics rollout.
* We ignore the `mean`/`std` rows stored in those files and recompute every table cell from the 2000 per-sequence rows.

See `README.md` next to this notebook for what each metric means.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE))
import l1_retargeting as L

L1 = L.l1_dir(HERE)
print("L1 results folder:", L1)
L.verify_checksums(L1)
print("sha256 of all", len(L.SHA256), "input files OK")

## 1. Load the per-sequence results

The row order of every report is the first 2000 entries of the test split `amass_humanoid_v2/full_test.json`, so we label each row with its sequence id.

In [ ]:
sids = L.test_sids(L1)
per_seq = {key: L.load_per_sequence(L1 / "reports", stem, sids) for key, stem in L.RUNS.items()}

cols = ["SR-seq", "SR-succ_n", "SR-total_n", "gerr", "lerr", "retgt_nframes", "retgt_runtime", "retgt_fps"]
print("WMM, clean setting, first 5 sequences:")
per_seq[("WMM", "clean")][cols].head()

## 2. Recompute the dataset-level numbers

For every (method, setting):

* **SR-seq** = fraction of sequences where the robot never fell (`SR-succ_n == SR-total_n`, i.e. every frame was a standing frame).
* **gerr / lerr** = average over sequences of the per-sequence tracking error (metres, standing frames only).
* **FPS** = average over sequences of `frames / retargeting wall-clock seconds`, using the run time logged for each sequence during inference.

In [ ]:
results = {key: L.recompute(df) for key, df in per_seq.items()}
pd.DataFrame(results).T.rename_axis(["method", "setting"])

## 3. The table, in the paper's layout

In [ ]:
table = L.paper_layout(results)
print(table.to_string())
table

## 4. Extra numbers not printed in the paper

The paper reports speed only for the clean setting. The same logs give speed for the other two settings. GMR in the in-fill setting retargets only the 3 FPS keyframes and interpolates, so its frames-per-second is high.

In [ ]:
pd.DataFrame({s: {m: round(results[(m, s)]["FPS"], 2) for m in L.METHODS} for s in L.SETTINGS})

## 5. Per-sequence view

The table hides how spread out the errors are. Each box shows the per-sequence local error `lerr` (centimetres) over the 2000 test clips; the dot is the mean that enters the table.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
colors = {"GMR": "#9aa5b1", "PHUMA": "#c9a227", "WMM": "#e07a5f"}
for ax, setting in zip(axes, L.SETTINGS):
    data = [100 * per_seq[(m, setting)]["lerr"].values for m in L.METHODS]
    box = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6)
    for patch, m in zip(box["boxes"], L.METHODS):
        patch.set_facecolor(colors[m])
    ax.plot(range(1, 4), [100 * results[(m, setting)]["lerr"] for m in L.METHODS], "ko", ms=4)
    labels = [f"{m}\nSR {results[(m, setting)]['SR-seq']:.4f}" for m in L.METHODS]
    ax.set_xticks(range(1, 4), labels)
    ax.set_title(L.SETTING_TITLES[setting])
axes[0].set_ylabel("lerr per sequence (cm)")
fig.tight_layout()
fig.savefig(HERE / "figures" / "l1_per_sequence_lerr.png", dpi=120)
plt.show()